In [1]:
import pandas as pd
master_products = pd.read_csv("../data/phase1/master_products.csv")

In [2]:
content_df = master_products[["product_id", "product_name", "category", "subcategory", "brand", "product_type", "description"]].copy()
content_df.head()

,product_id,product_name,category,subcategory,brand,product_type,description
0,P000001,Garlic Oil - Vegetarian Capsule 500 mg,Beauty & Hygiene,Hair Care,Sri Sri Ayurveda,Hair Oil & Serum,This Product contains Garlic Oil that is known...
1,P000002,Water Bottle - Orange,"Kitchen, Garden & Pets",Storage & Accessories,Mastercook,Water & Fridge Bottles,"Each product is microwave safe (without lid), ..."
2,P000003,"Brass Angle Deep - Plain, No.2",Cleaning & Household,Pooja Needs,Trm,Lamp & Lamp Oil,"A perfect gift for all occasions, be it your m..."
3,P000004,Cereal Flip Lid Container/Storage Jar - Assort...,Cleaning & Household,Bins & Bathroom Ware,Nakoda,"Laundry, Storage Baskets",Multipurpose container with an attractive desi...
4,P000005,Creme Soft Soap - For Hands & Body,Beauty & Hygiene,Bath & Hand Wash,Nivea,Bathing Bars & Soaps,Nivea Creme Soft Soap gives your skin the best...


In [3]:
content_df = content_df.fillna("")
content_df.isna().sum()

product_id      0
product_name    0
category        0
subcategory     0
brand           0
product_type    0
description     0
dtype: int64

In [4]:
content_df["combined_features"] = (
    content_df["category"] + " " + 
    content_df["subcategory"] + " " +
    content_df["brand"] + " " +
    content_df["product_type"] + " " +
    content_df["description"]
)
content_df[["product_name", "combined_features"]].head()

,product_name,combined_features
0,Garlic Oil - Vegetarian Capsule 500 mg,Beauty & Hygiene Hair Care Sri Sri Ayurveda Ha...
1,Water Bottle - Orange,"Kitchen, Garden & Pets Storage & Accessories M..."
2,"Brass Angle Deep - Plain, No.2",Cleaning & Household Pooja Needs Trm Lamp & La...
3,Cereal Flip Lid Container/Storage Jar - Assort...,Cleaning & Household Bins & Bathroom Ware Nako...
4,Creme Soft Soap - For Hands & Body,Beauty & Hygiene Bath & Hand Wash Nivea Bathin...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english"
)
tfidf_matrix = tfidf.fit_transform(content_df["combined_features"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (27555, 29088)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(tfidf_matrix)
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (27555, 27555)


In [26]:
import numpy as np

product_indices = pd.Series(
    content_df.index,
    index=content_df["product_name"]
).drop_duplicates()

def recommend_products(product_name, n=5):
    matching_indices = content_df.index[content_df["product_name"] == product_name].tolist()
    if len(matching_indices) == 0:
        return "Product not found"
    idx = matching_indices[0]
    similarity_scores = similarity_matrix[idx]
    similar_indices = [i for i in np.argsort(similarity_scores)[::-1] if content_df.iloc[i]["product_name"] != product_name][:n]
    recommendations = content_df.iloc[similar_indices].copy()
    recommendations["similarity_score"] = [similarity_scores[i] for i in similar_indices]
    return recommendations

In [27]:
content_df["product_name"].iloc[0]

'Garlic Oil - Vegetarian Capsule 500 mg'

In [22]:
content_df.columns.tolist()

['product_id',
 'product_name',
 'category',
 'subcategory',
 'brand',
 'product_type',
 'description',
 'combined_features']

In [28]:
recommend_products(
    "Garlic Oil - Vegetarian Capsule 500 mg", n=5
) [["product_name", "category", "brand", "similarity_score"]]

,product_name,category,brand,similarity_score
10559,Brahmi Bhringaraj Taila - Anti Graying,Beauty & Hygiene,Sri Sri Ayurveda,0.514698
766,Evening Primrose Oil - Vegetarian Capsule (500...,Beauty & Hygiene,Sri Sri Ayurveda,0.492513
22164,Hair Oil - Amla,Beauty & Hygiene,Patanjali,0.461927
24044,Coconut Oil - 100 % Pure,Beauty & Hygiene,Parachute,0.440509
16359,"Flaxseed Oil - Omega-3, Omega-6, Omega-9 Veget...",Beauty & Hygiene,Sri Sri Ayurveda,0.435621


In [29]:
import joblib
from scipy.sparse import save_npz

joblib.dump(tfidf, "../models/content_based/tfidf_vectorizer.pkl")
save_npz("../models/content_based/tfidf_matrix.npz", tfidf_matrix)
content_df.to_csv("../models/content_based/content_products.csv", index=False)
print("Model saved successfully...")

Model saved successfully...
